# Reformat SY Export

### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
from typing import Tuple

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm  # Progress bar

import climakitae as ck
from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
    set_profile_metadata,
    _format_based_on_structure,
    _construct_profile_dataframe,
    _create_simple_dataframe,
    _create_single_wl_multi_sim_dataframe,
    _create_multi_wl_single_sim_dataframe,
    _create_multi_wl_multi_sim_dataframe,
     _stack_profile_data
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.core.data_interface import (
    get_data_options,
    get_subsetting_options,
    get_data,
)

import warnings
warnings.filterwarnings("ignore")


from unittest.mock import MagicMock, call, patch
import pytest

## Where is the issue?

In [61]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5]
simulations = ["sim1"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

sample_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)


In [62]:
sample_data

<xarray.DataArray (warming_level: 1, time_delta: 8760, simulation: 1)> Size: 70kB
array([[[6.63941289e-04],
        [8.91040257e-01],
        [4.16001929e-01],
        ...,
        [9.54159875e-01],
        [6.28306409e-02],
        [8.39508647e-01]]])
Coordinates:
  * warming_level  (warming_level) float64 8B 1.5
  * time_delta     (time_delta) datetime64[ns] 70kB 2020-01-01 ... 2020-12-30...
  * simulation     (simulation) <U4 16B 'sim1'
Attributes:
    units:        degC
    variable_id:  tasmax

In [52]:
days_in_year = 365
hours_in_year = 8760

In [53]:
def og_compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                # Reshape to (days_in_year, 24) for the final DataFrame
                profile_reshaped = profile_1d[: days_in_year * hours_per_day].reshape(
                    days_in_year, hours_per_day
                )

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_reshaped

                pbar.update(1)

    # Create the multi-index DataFrame structure
    df_profile = _construct_profile_dataframe(
        profile_data=profile_data,
        warming_levels=warming_levels,
        simulations=simulations,
        sim_label_func=_get_simulation_label,
        days_in_year=days_in_year,
        hours_per_day=hours_per_day,
    )

    # Determine which formatting function to use based on the structure
    _format_based_on_structure(df_profile)

    # Prepare metadata dictionary
    metadata = {
        "quantile": q,
        "method": "8760 analysis - actual data closest to quantile across 30 years",
        "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    }

    # Add original data attributes if available
    if hasattr(data, "attrs"):
        if "units" in data.attrs:
            metadata["units"] = data.attrs["units"]
        if "extended_description" in data.attrs:
            metadata["extended_description"] = data.attrs["extended_description"]
        if "variable_id" in data.attrs:
            metadata["variable_name"] = data.attrs["variable_id"]
        elif hasattr(data, "name") and data.name:
            metadata["variable_name"] = data.name

    # Set all metadata using the helper function
    set_profile_metadata(df_profile, metadata)

    print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    print(
        f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    )
    if hasattr(data, "attrs") and "units" in data.attrs:
        print(f"         Units: {data.attrs['units']}")

    return df_profile


In [57]:
og_result = og_compute_profile(sample_data, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/1 [00:00<?, ?combo/s]

n_warming_levels: 1
n_simulations: 1
      ✅ Profile computation complete! Final shape: (365, 24)
         With index: Day of Year, columns: ['Hour', 'Simulation']
         Units: degC


In [58]:
og_result

Hour,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
Simulation,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,...,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1
Day of Year,,,,,,,,,,,,,,,,,,,,,
Jan-01,0.487594,0.166068,0.744244,0.778232,0.100737,0.552811,0.922903,0.852127,0.700669,0.776527,...,0.390065,0.140933,0.966132,0.971132,0.100872,0.432763,0.549089,0.094597,0.516972,0.061723
Jan-02,0.684216,0.842156,0.337727,0.481706,0.634274,0.447854,0.679247,0.339323,0.286843,0.956199,...,0.462088,0.628516,0.570566,0.581387,0.336548,0.269604,0.511959,0.813381,0.398851,0.395382
Jan-03,0.634091,0.186755,0.832655,0.342141,0.973036,0.173640,0.578908,0.019062,0.377247,0.621497,...,0.533708,0.178079,0.445053,0.867751,0.046432,0.083945,0.285092,0.680269,0.118938,0.474695
Jan-04,0.633686,0.022096,0.802039,0.101948,0.141568,0.819833,0.302570,0.515514,0.364949,0.039523,...,0.330129,0.036978,0.022305,0.652435,0.151021,0.533368,0.417479,0.972733,0.886826,0.372576
Jan-05,0.347647,0.223570,0.923018,0.268353,0.266789,0.789637,0.307259,0.061512,0.493912,0.143855,...,0.303942,0.509317,0.199013,0.212024,0.789864,0.486129,0.678299,0.024303,0.461691,0.516840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Dec-27,0.831103,0.759554,0.763666,0.429490,0.331627,0.386670,0.558783,0.269176,0.334419,0.781762,...,0.834638,0.475768,0.897467,0.860542,0.499241,0.431421,0.732780,0.608482,0.963840,0.743958
Dec-28,0.796698,0.144007,0.460471,0.856839,0.355471,0.617700,0.370093,0.549075,0.991470,0.517261,...,0.609534,0.352617,0.673872,0.939575,0.273991,0.513528,0.150835,0.457898,0.881474,0.427564


In [4]:
# Check for simulation dimension
has_simulation = "simulation" in sample_data.dims
if has_simulation:
    n_simulations = len(sample_data.simulation)
    simulations = sample_data.simulation.values
else:
    n_simulations = 1
    simulations = [None]

# Get all available time_delta data (all 30 years)
hours_per_day = 24
hours_per_year = 8760
total_hours = len(sample_data.time_delta)
n_years = total_hours // hours_per_year

warming_levels = sample_data.warming_level.values

# Create helper function to extract meaningful simulation labels
def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
    """Extract meaningful simulation label from simulation identifier."""
    if sim is None:
        return f"Sim_{sim_idx+1}"

    sim_str = str(sim)
    if "WRF_" in sim_str:
        # Parse simulation name format: WRF_GCM_params_scenario
        # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
        parts = sim_str.split("_")
        if len(parts) >= 4:
            gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
            params = parts[2]  # e.g., r11i1p1f1
            scenario = parts[3]  # e.g., historical+ssp245

            # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
            if "ssp" in scenario:
                ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                ssp = f"ssp{ssp_part}"
            else:
                ssp = "hist"  # fallback for historical-only

            return f"{gcm}-{params}-{ssp}"
        elif len(parts) >= 2:
            # Fallback for shorter format
            return f"{parts[1]}-{sim_idx+1}"
        else:
            return f"Sim_{sim_idx+1}"
    else:
        # Ensure uniqueness by adding index for non-WRF format
        base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
        return f"{base_name}-{sim_idx+1}"

In [66]:
def compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}
    #! -start
    profile_data_reshaped = {}
    profile_data_1d = {}
    #! -end

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                # Reshape to (days_in_year, 24) for the final DataFrame
                profile_reshaped = profile_1d[: days_in_year * hours_per_day].reshape(
                    days_in_year, hours_per_day
                )

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_reshaped
                #! -start
                profile_data_reshaped[key] = profile_reshaped
                profile_data_1d[key] = profile_1d
                #! -end

                pbar.update(1)

    # # Create the multi-index DataFrame structure
    # df_profile = _construct_profile_dataframe(
    #     profile_data=profile_data,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -start
    # df_profile_reshaped = _construct_profile_dataframe(
    #     profile_data=profile_data_reshaped,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # df_profile_1d = _construct_profile_dataframe(
    #     profile_data=profile_data_1d,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -end

    # # Determine which formatting function to use based on the structure
    # _format_based_on_structure(df_profile)

    # #! -start
    # _format_based_on_structure(df_profile_reshaped)
    # _format_based_on_structure(df_profile_1d)
    # #! -end

    # # Prepare metadata dictionary
    # metadata = {
    #     "quantile": q,
    #     "method": "8760 analysis - actual data closest to quantile across 30 years",
    #     "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    # }

    # # Add original data attributes if available
    # if hasattr(data, "attrs"):
    #     if "units" in data.attrs:
    #         metadata["units"] = data.attrs["units"]
    #     if "extended_description" in data.attrs:
    #         metadata["extended_description"] = data.attrs["extended_description"]
    #     if "variable_id" in data.attrs:
    #         metadata["variable_name"] = data.attrs["variable_id"]
    #     elif hasattr(data, "name") and data.name:
    #         metadata["variable_name"] = data.name

    # # Set all metadata using the helper function
    # set_profile_metadata(df_profile, metadata)

    # #! -start
    # set_profile_metadata(df_profile_reshaped, metadata)
    # set_profile_metadata(df_profile_1d, metadata)
    # #! -end

    # print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    # print(
    #     f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    # )
    # if hasattr(data, "attrs") and "units" in data.attrs:
    #     print(f"         Units: {data.attrs['units']}")

    return profile_data_reshaped, profile_data_1d

In [6]:
profile_data_reshaped, profile_data_1d = compute_profile(sample_data, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/1 [00:00<?, ?combo/s]

In [49]:
profile_data_reshaped

{('WL_1.5',
  'sim1-1'): array([[0.00235343, 0.26512841, 0.88235051, ..., 0.0742842 , 0.91593185,
         0.4321042 ],
        [0.71687669, 0.86894677, 0.21255754, ..., 0.57248272, 0.90405073,
         0.68000251],
        [0.80855521, 0.12684446, 0.44241673, ..., 0.55909537, 0.90542529,
         0.48662585],
        ...,
        [0.57768075, 0.6640151 , 0.08985435, ..., 0.09555943, 0.52421779,
         0.96922263],
        [0.42452817, 0.22064797, 0.28533868, ..., 0.36581027, 0.82019198,
         0.94478128],
        [0.95369268, 0.61815687, 0.47353772, ..., 0.30486521, 0.00757574,
         0.62881162]])}

In [7]:
sub_reshaped = profile_data_reshaped[('WL_1.5',
  'sim1-1')]
print(len(sub_reshaped[1]))
print(len(profile_data_reshaped[('WL_1.5',
  'sim1-1')]))
sub_1d = profile_data_1d[('WL_1.5',
  'sim1-1')]
print(len(sub_1d))

24
365
8760


profile_data_1d is 8760 long, while profile_data_reshaped is a set of 365 arrays, each 24 long. 

In [8]:
def _construct_profile_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulations: list,
    sim_label_func: callable,
    days_in_year: int,
    hours_per_day: int,
) -> pd.DataFrame:
    """
    Construct a DataFrame from profile data based on warming level and simulation dimensions.

    Parameters
    ----------
    profile_data : dict
        Dictionary with (warming_level, simulation) keys and profile arrays as values
    warming_levels : np.ndarray
        Array of warming level values
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to extract simulation labels
    days_in_year : int
        Number of days in the year (365 or 366)
    hours_per_day : int
        Number of hours per day (24)

    Returns
    -------
    pd.DataFrame
        Structured DataFrame with appropriate column structure based on dimensions
    """
    hours = np.arange(1, 25, 1)
    n_warming_levels = len(warming_levels)
    n_simulations = len(simulations)
    print(f"n_warming_levels: {n_warming_levels}")
    print(f"n_simulations: {n_simulations}")

    if n_warming_levels == 1 and n_simulations == 1:
        return _create_simple_dataframe(
            profile_data,
            warming_levels[0],
            simulations[0],
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
        )
    # elif n_warming_levels == 1 and n_simulations > 1:
    #     return _create_single_wl_multi_sim_dataframe(
    #         profile_data,
    #         warming_levels[0],
    #         simulations,
    #         sim_label_func,
    #         days_in_year,
    #         hours,
    #         hours_per_day,
    #     )
    # elif n_warming_levels > 1 and n_simulations == 1:
    #     return _create_multi_wl_single_sim_dataframe(
    #         profile_data,
    #         warming_levels,
    #         simulations[0],
    #         sim_label_func,
    #         days_in_year,
    #         hours,
    #         hours_per_day,
    #     )
    # else:
    #     return _create_multi_wl_multi_sim_dataframe(
    #         profile_data,
    #         warming_levels,
    #         simulations,
    #         sim_label_func,
    #         days_in_year,
    #         hours,
    #         hours_per_day,
    #     )


In [9]:
# Create the multi-index DataFrame structure
df_profile_reshaped = _construct_profile_dataframe(
    profile_data=profile_data_reshaped,
    warming_levels=warming_levels,
    simulations=simulations,
    sim_label_func=_get_simulation_label,
    days_in_year=days_in_year,
    hours_per_day=hours_per_day,
)

n_warming_levels: 1
n_simulations: 1


In [10]:
profile_data=profile_data_reshaped
warming_levels=warming_levels
simulations=simulations
sim_label_func=_get_simulation_label
days_in_year=days_in_year
hours_per_day=hours_per_day
hours = np.arange(1, 25, 1)

### The root of the problem
_create_simple_dataframe, and its ilk, will need to be modified.

In [11]:
def _create_simple_dataframe(
    profile_data: dict,
    warming_level: float,
    simulation: any,
    sim_label_func: callable,
    days_in_year: int,
    hours: np.ndarray,
    hours_per_day: int,
) -> pd.DataFrame:
    """
    Create a simple DataFrame for single warming level and single simulation.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_level : float
        Single warming level value
    simulation : any
        Single simulation identifier
    sim_label_func : callable
        Function to get simulation label
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        Simple DataFrame with hour columns
    """

    wl_key = warming_level
    sim_key = sim_label_func(simulation, 0)

    # Create MultiIndex columns
    col_tuples = [(hour, sim_key) for hour in hours]
    multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Simulation"])

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        hours_per_day=hours_per_day,
        wl_names=[f"WL_{wl_key}"],
        sim_names=[sim_key],
        hour_first=True,
    )

    return pd.DataFrame(
        all_data,
        columns=multi_cols,
        index=np.arange(1, days_in_year + 1, 1),
    )


In [12]:
test_profile = _create_simple_dataframe(
            profile_data,
            warming_levels[0],
            simulations[0],
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
        )

Exploding _create_simple_dataframe()

In [13]:
profile_data = profile_data_reshaped
#profile_data = profile_data_1d
warming_level = warming_levels[0]
simulation = simulations[0]
sim_label_func = sim_label_func
days_in_year = days_in_year
hours = hours
hours_per_day = hours_per_day 

In [14]:
wl_key = warming_level
sim_key = sim_label_func(simulation, 0)

# Create MultiIndex columns
col_tuples = [(hour, sim_key) for hour in hours]
multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Simulation"])

# Stack data
all_data = _stack_profile_data(
    profile_data=profile_data,
    hours_per_day=hours_per_day,
    wl_names=[f"WL_{wl_key}"],
    sim_names=[sim_key],
    hour_first=True,
)

pd.DataFrame(
    all_data,
    columns=multi_cols,
    index=np.arange(1, days_in_year + 1, 1),
)

Hour,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
Simulation,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,...,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1
1,0.002353,0.265128,0.882351,0.169491,0.958451,0.047984,0.244131,0.986262,0.561196,0.442382,...,0.253784,0.715774,0.799341,0.691457,0.466091,0.034453,0.219753,0.074284,0.915932,0.432104
2,0.716877,0.868947,0.212558,0.022783,0.159640,0.858707,0.115825,0.218340,0.351020,0.219893,...,0.055705,0.004096,0.475815,0.420878,0.689193,0.204393,0.345996,0.572483,0.904051,0.680003
3,0.808555,0.126844,0.442417,0.853251,0.596914,0.888314,0.304940,0.600085,0.665168,0.600223,...,0.437830,0.625908,0.075623,0.544075,0.087211,0.773301,0.425526,0.559095,0.905425,0.486626
4,0.896302,0.449592,0.486266,0.353818,0.924219,0.583488,0.154939,0.445904,0.738571,0.594030,...,0.885455,0.256832,0.973245,0.106744,0.615650,0.707219,0.154143,0.813790,0.957153,0.632845
5,0.984320,0.112624,0.214333,0.560196,0.903513,0.389719,0.309054,0.575866,0.616296,0.865132,...,0.670735,0.073219,0.828710,0.798983,0.296458,0.254312,0.240605,0.281497,0.370786,0.594202
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,0.018441,0.899365,0.438538,0.713936,0.334387,0.490543,0.924917,0.176731,0.076169,0.274450,...,0.300120,0.209428,0.965452,0.290570,0.260912,0.544519,0.131919,0.086130,0.448708,0.794257
362,0.616365,0.656435,0.993643,0.308084,0.229254,0.994081,0.903999,0.518789,0.255209,0.849866,...,0.772978,0.375974,0.382873,0.020413,0.857125,0.703748,0.871151,0.311251,0.383126,0.280108
363,0.577681,0.664015,0.089854,0.656608,0.478216,0.272054,0.072078,0.261451,0.306451,0.850115,...,0.592694,0.584004,0.105665,0.736625,0.003215,0.033666,0.071808,0.095559,0.524218,0.969223


Now to explode _stack_profile_data

In [42]:
profile_data = profile_data_reshaped
#profile_data = profile_data_1d
hours_per_day=hours_per_day
wl_names=[f"WL_{wl_key}"]
sim_names=[sim_key]
hour_first = True,
three_level = False

In [43]:
# all_data = []

# if three_level:
#     # For three-level index: iterate hour -> wl -> sim
#     for hour in range(hours_per_day):
#         for wl_name in wl_names:
#             for sim_name in sim_names:
#                 key = (wl_name, sim_name)
#                 all_data.append(profile_data[key][:, hour])
# elif hour_first:
#     # For two-level with hour first
#     for hour in range(hours_per_day):
#         for wl_name in wl_names:
#             for sim_name in sim_names:
#                 key = (wl_name, sim_name)
#                 all_data.append(profile_data[key][:, hour])
# else:
#     # For other two-level cases
#     for wl_name in wl_names:
#         for sim_name in sim_names:
#             for hour in range(hours_per_day):
#                 key = (wl_name, sim_name)
#                 all_data.append(profile_data[key][:, hour])

# np.column_stack(all_data)

In [44]:
all_data = []

for hour in range(hours_per_day):
    for wl_name in wl_names:
        for sim_name in sim_names:
            key = (wl_name, sim_name)
            all_data.append(profile_data[key][:, hour])

In [45]:
len(all_data[1])

365

In [46]:
stacked_data = np.column_stack(all_data)

In [47]:
final_df = pd.DataFrame(
    stacked_data,
    columns=multi_cols,
    index=np.arange(1, days_in_year + 1, 1),
)

In [48]:
final_df

Hour,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
Simulation,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,...,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1
1,0.002353,0.265128,0.882351,0.169491,0.958451,0.047984,0.244131,0.986262,0.561196,0.442382,...,0.253784,0.715774,0.799341,0.691457,0.466091,0.034453,0.219753,0.074284,0.915932,0.432104
2,0.716877,0.868947,0.212558,0.022783,0.159640,0.858707,0.115825,0.218340,0.351020,0.219893,...,0.055705,0.004096,0.475815,0.420878,0.689193,0.204393,0.345996,0.572483,0.904051,0.680003
3,0.808555,0.126844,0.442417,0.853251,0.596914,0.888314,0.304940,0.600085,0.665168,0.600223,...,0.437830,0.625908,0.075623,0.544075,0.087211,0.773301,0.425526,0.559095,0.905425,0.486626
4,0.896302,0.449592,0.486266,0.353818,0.924219,0.583488,0.154939,0.445904,0.738571,0.594030,...,0.885455,0.256832,0.973245,0.106744,0.615650,0.707219,0.154143,0.813790,0.957153,0.632845
5,0.984320,0.112624,0.214333,0.560196,0.903513,0.389719,0.309054,0.575866,0.616296,0.865132,...,0.670735,0.073219,0.828710,0.798983,0.296458,0.254312,0.240605,0.281497,0.370786,0.594202
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,0.018441,0.899365,0.438538,0.713936,0.334387,0.490543,0.924917,0.176731,0.076169,0.274450,...,0.300120,0.209428,0.965452,0.290570,0.260912,0.544519,0.131919,0.086130,0.448708,0.794257
362,0.616365,0.656435,0.993643,0.308084,0.229254,0.994081,0.903999,0.518789,0.255209,0.849866,...,0.772978,0.375974,0.382873,0.020413,0.857125,0.703748,0.871151,0.311251,0.383126,0.280108
363,0.577681,0.664015,0.089854,0.656608,0.478216,0.272054,0.072078,0.261451,0.306451,0.850115,...,0.592694,0.584004,0.105665,0.736625,0.003215,0.033666,0.071808,0.095559,0.524218,0.969223


Now to massage to handle profile_data_1d

In [38]:
wl_names[0]

'WL_1.5'

In [37]:
sim_names

['sim1-1']

In [39]:

final_df = pd.DataFrame(
    profile_data_1d[(wl_names[0],
  sim_names[0])],
    columns=[sim_key],
    index=np.arange(1, hours_in_year + 1, 1),
)

In [ ]:
# we want the final result to look like this
final_df

,sim1-1
1,0.002353
2,0.265128
3,0.882351
4,0.169491
5,0.958451
...,...
8756,0.050514
8757,0.628289
8758,0.304865
8759,0.007576


#### Beyond simple dataframes

In [64]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5]
simulations = ["sim1","sim2"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

sample_data_multi_sim = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

In [70]:
profile_data_reshaped_multi_sim, profile_data_1d_multi_sim = compute_profile(sample_data_multi_sim, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

#### Exploded final functions

Alright, I see it now. Let's go from the top.

In [ ]:
# the original result looks like this
og_result

Hour,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
Simulation,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,...,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1
Day of Year,,,,,,,,,,,,,,,,,,,,,
Jan-01,0.487594,0.166068,0.744244,0.778232,0.100737,0.552811,0.922903,0.852127,0.700669,0.776527,...,0.390065,0.140933,0.966132,0.971132,0.100872,0.432763,0.549089,0.094597,0.516972,0.061723
Jan-02,0.684216,0.842156,0.337727,0.481706,0.634274,0.447854,0.679247,0.339323,0.286843,0.956199,...,0.462088,0.628516,0.570566,0.581387,0.336548,0.269604,0.511959,0.813381,0.398851,0.395382
Jan-03,0.634091,0.186755,0.832655,0.342141,0.973036,0.173640,0.578908,0.019062,0.377247,0.621497,...,0.533708,0.178079,0.445053,0.867751,0.046432,0.083945,0.285092,0.680269,0.118938,0.474695
Jan-04,0.633686,0.022096,0.802039,0.101948,0.141568,0.819833,0.302570,0.515514,0.364949,0.039523,...,0.330129,0.036978,0.022305,0.652435,0.151021,0.533368,0.417479,0.972733,0.886826,0.372576
Jan-05,0.347647,0.223570,0.923018,0.268353,0.266789,0.789637,0.307259,0.061512,0.493912,0.143855,...,0.303942,0.509317,0.199013,0.212024,0.789864,0.486129,0.678299,0.024303,0.461691,0.516840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Dec-27,0.831103,0.759554,0.763666,0.429490,0.331627,0.386670,0.558783,0.269176,0.334419,0.781762,...,0.834638,0.475768,0.897467,0.860542,0.499241,0.431421,0.732780,0.608482,0.963840,0.743958
Dec-28,0.796698,0.144007,0.460471,0.856839,0.355471,0.617700,0.370093,0.549075,0.991470,0.517261,...,0.609534,0.352617,0.673872,0.939575,0.273991,0.513528,0.150835,0.457898,0.881474,0.427564


In [ ]:
def compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                #! removed the reshaping
                # # Reshape to (days_in_year, 24) for the final DataFrame
                # profile_reshaped = profile_1d[: days_in_year * hours_per_day].reshape(
                #     days_in_year, hours_per_day
                # )

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_1d

                pbar.update(1)

    ###################################################################
    ###################################################################
    ###################################################################
    # Create the multi-index DataFrame structure

    #! explode each element 

    #!#!  start _construct_profile_dataframe
    sim_label_func=_get_simulation_label
    # df_profile = _construct_profile_dataframe(
    #     profile_data=profile_data,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )

    hours = np.arange(1, 25, 1)
    n_warming_levels = len(warming_levels)
    n_simulations = len(simulations)

    if n_warming_levels == 1 and n_simulations == 1:
        #!#!#! start  _create_simple_dataframe
        warming_level = warming_levels[0]
        simulation = simulations[0]
        # return _create_simple_dataframe(
        #             profile_data,
        #             warming_levels[0],
        #             simulations[0],
        #             sim_label_func,
        #             days_in_year,
        #             hours,
        #             hours_per_day,
        #         )


        wl_key = warming_level
        sim_key = sim_label_func(simulation, 0)

        # Create MultiIndex columns
        # col_tuples = [(hour, sim_key) for hour in hours]
        # multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Simulation"])

        # Stack data
        #!#!#!#! start  _stack_profile_data
        wl_names = [f"WL_{wl_key}"]
        sim_names = [sim_key]
        hour_first = True
        three_level = False

        # all_data = _stack_profile_data(
        #     profile_data=profile_data,
        #     hours_per_day=hours_per_day,
        #     wl_names=[f"WL_{wl_key}"],
        #     sim_names=[sim_key],
        #     hour_first=True,
        # )
        all_data = []

        if three_level:
            # For three-level index: iterate hour -> wl -> sim
            for hour in range(hours_per_day):
                for wl_name in wl_names:
                    for sim_name in sim_names:
                        key = (wl_name, sim_name)
                        all_data.append(profile_data[key][:, hour])
        elif hour_first:
            # For two-level with hour first
            for hour in range(hours_per_day):
                for wl_name in wl_names:
                    for sim_name in sim_names:
                        key = (wl_name, sim_name)
                        all_data.append(profile_data[key][:, hour])
        else:
            # For other two-level cases
            for wl_name in wl_names:
                for sim_name in sim_names:
                    for hour in range(hours_per_day):
                        key = (wl_name, sim_name)
                        all_data.append(profile_data[key][:, hour])

        all_data =  np.column_stack(all_data)

        #!#!#!#! end _stack_profile_data

        # pd.DataFrame(
        #     all_data,
        #     columns=multi_cols,
        #     index=np.arange(1, days_in_year + 1, 1),
        # )

        df_profile = pd.DataFrame(
            profile_data_1d[(wl_names[0],
            sim_names[0])],
            columns=[sim_key],
            index=np.arange(1, hours_in_year + 1, 1),
        )
        #!#!#! end  _create_simple_dataframe

    elif n_warming_levels == 1 and n_simulations > 1:
        #!#!#! start  _create_single_wl_multi_sim_dataframe
        warming_level = warming_levels[0]

        # df_profile = _create_single_wl_multi_sim_dataframe(
        #     profile_data,
        #     warming_levels[0],
        #     simulations,
        #     sim_label_func,
        #     days_in_year,
        #     hours,
        #     hours_per_day,
        # )
        
        wl = warming_level
        sim_names = [sim_label_func(sim, i) for i, sim in enumerate(simulations)]

        # Ensure simulation names are unique
        if len(sim_names) != len(set(sim_names)):
            print(
                "   ⚠️  Warning: Duplicate simulation names detected, adding uniqueness suffixes"
            )
            unique_sim_names = []
            name_counts = {}
            for name in sim_names:
                if name not in name_counts:
                    name_counts[name] = 0
                    unique_sim_names.append(name)
                else:
                    name_counts[name] += 1
                    unique_sim_names.append(f"{name}_v{name_counts[name]}")
            sim_names = unique_sim_names

        # Create MultiIndex columns
        col_tuples = [(hour, sim_name) for hour in hours for sim_name in sim_names]
        multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Simulation"])

        # Stack data
        all_data = _stack_profile_data(
            profile_data=profile_data,
            hours_per_day=hours_per_day,
            wl_names=[f"WL_{wl}"],
            sim_names=sim_names,
            hour_first=True,
        )

        return pd.DataFrame(
            all_data,
            columns=multi_cols,
            index=np.arange(1, days_in_year + 1, 1),
        )

        


        #!#!#! end  _create_single_wl_multi_sim_dataframe
    elif n_warming_levels > 1 and n_simulations == 1:
        df_profile = _create_multi_wl_single_sim_dataframe(
            profile_data,
            warming_levels,
            simulations[0],
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
        )
    else:
        df_profile = _create_multi_wl_multi_sim_dataframe(
            profile_data,
            warming_levels,
            simulations,
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
        )
    #!#!  start _construct_profile_dataframe

    ###################################################################
    ###################################################################
    ###################################################################

    # #! -start
    # df_profile_reshaped = _construct_profile_dataframe(
    #     profile_data=profile_data_reshaped,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # df_profile_1d = _construct_profile_dataframe(
    #     profile_data=profile_data_1d,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -end

    # # Determine which formatting function to use based on the structure
    # _format_based_on_structure(df_profile)

    # #! -start
    # _format_based_on_structure(df_profile_reshaped)
    # _format_based_on_structure(df_profile_1d)
    # #! -end

    # # Prepare metadata dictionary
    # metadata = {
    #     "quantile": q,
    #     "method": "8760 analysis - actual data closest to quantile across 30 years",
    #     "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    # }

    # # Add original data attributes if available
    # if hasattr(data, "attrs"):
    #     if "units" in data.attrs:
    #         metadata["units"] = data.attrs["units"]
    #     if "extended_description" in data.attrs:
    #         metadata["extended_description"] = data.attrs["extended_description"]
    #     if "variable_id" in data.attrs:
    #         metadata["variable_name"] = data.attrs["variable_id"]
    #     elif hasattr(data, "name") and data.name:
    #         metadata["variable_name"] = data.name

    # # Set all metadata using the helper function
    # set_profile_metadata(df_profile, metadata)

    # #! -start
    # set_profile_metadata(df_profile_reshaped, metadata)
    # set_profile_metadata(df_profile_1d, metadata)
    # #! -end

    # print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    # print(
    #     f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    # )
    # if hasattr(data, "attrs") and "units" in data.attrs:
    #     print(f"         Units: {data.attrs['units']}")

    return profile_data_reshaped, profile_data_1d

## Testing

In [ ]:
# Set up the Standard Year generator
profile_selections = {  
    ## Required variable and profile arguments
    "variable": "Air Temperature at 2m",
    "resolution": "3 km",
    "q": 0.5,
    "units": "degF",

    ## Required approach arguments, Options: "Warming Level", "Time"
    "approach": "Warming Level",
    # "warming_level": [warming_level], # GWL-option only
    # "centered_year": centered_year, # Time-based option only
    
    ## Required location arguments -- Uncomment your desired location option and modify
    "stations": ["Sacramento Executive Airport (KSAC)"], 
    # "latitude": (latitude - 0.02, latitude + 0.02), 
    # "longitude": (longitude - 0.02, longitude + 0.02),  
    # "cached_area": area_name, 

    ## Additional optional arguments -- Uncomment any desired options and modify
    # "no_delta": True, 
    # "warming_level_window": 10,
    # "time_profile_scenario": "SSP2-4.5,
    # "bias_adjusted_models": True,
}

# Generate the climate profile
profile = get_climate_profile(**profile_selections)

In [ ]:
# the function uses the previously defined profile selections to generate the output file name
export_profile_to_csv(profile, **profile_selections)